In [1]:
# Download the dataset from kaggle

!wget https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
!unzip ml-latest-small.zip

--2026-06-14 16:04:07--  https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 978202 (955K) [application/zip]
Saving to: ‘ml-latest-small.zip’

ml-latest-small.zip 100%[===================>] 955.28K  2.82MB/s    in 0.3s    

2026-06-14 16:04:08 (2.82 MB/s) - ‘ml-latest-small.zip’ saved [978202/978202]

Archive:  ml-latest-small.zip
   creating: ml-latest-small/
  inflating: ml-latest-small/links.csv  
  inflating: ml-latest-small/tags.csv  
  inflating: ml-latest-small/ratings.csv  
  inflating: ml-latest-small/README.txt  
  inflating: ml-latest-small/movies.csv  


# Import Libraries

In [2]:

import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity

# Load Dataset

In [3]:
movies = pd.read_csv('/content/ml-latest-small/movies.csv')
ratings = pd.read_csv('/content/ml-latest-small/ratings.csv')

print(movies.head())
print(ratings.head())

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931


# Create User-Movie Matrix

Rows = Users
Columns = Movies
Values = Ratings

In [4]:

movie_matrix = ratings.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
)

movie_matrix.fillna(0, inplace=True)

movie_matrix.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,0.0,4.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# Calculate Similarity Matrix

In [5]:

movie_similarity = cosine_similarity(movie_matrix.T)

movie_similarity_df = pd.DataFrame(
    movie_similarity,
    index=movie_matrix.columns,
    columns=movie_matrix.columns
)

movie_similarity_df.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
movieId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.410562,0.296917,0.035573,0.308762,0.376316,0.277491,0.131629,0.232586,0.395573,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.410562,1.000000,0.282438,0.106415,0.287795,0.297009,0.228576,0.172498,0.044835,0.417693,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.296917,0.282438,1.000000,0.092406,0.417802,0.284257,0.402831,0.313434,0.304840,0.242954,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.035573,0.106415,0.092406,1.000000,0.188376,0.089685,0.275035,0.158022,0.000000,0.095598,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.308762,0.287795,0.417802,0.188376,1.000000,0.298969,0.474002,0.283523,0.335058,0.218061,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# Recommendations Function

In [6]:

def recommend_movies(movie_title, n=10):

    movie = movies[movies['title'] == movie_title]

    if movie.empty:
        return "Movie not found."

    movie_id = movie.iloc[0]['movieId']

    similar_movies = movie_similarity_df[movie_id]

    recommendations = similar_movies.sort_values(
        ascending=False
    )[1:n+1]

    recommended_titles = []

    for mid in recommendations.index:
        title = movies[movies['movieId'] == mid]['title'].values[0]
        recommended_titles.append(title)

    return recommended_titles

# Test the System

In [7]:
recommend_movies("Toy Story (1995)")

['Toy Story 2 (1999)',
 'Jurassic Park (1993)',
 'Independence Day (a.k.a. ID4) (1996)',
 'Star Wars: Episode IV - A New Hope (1977)',
 'Forrest Gump (1994)',
 'Lion King, The (1994)',
 'Star Wars: Episode VI - Return of the Jedi (1983)',
 'Mission: Impossible (1996)',
 'Groundhog Day (1993)',
 'Back to the Future (1985)']

# Evaluating the model

In [8]:

from sklearn.model_selection import train_test_split

train, test = train_test_split(
    ratings,
    test_size=0.2,
    random_state=42
)

print("Training Records:", len(train))
print("Testing Records:", len(test))

Training Records: 80668
Testing Records: 20168


# Advanced Version Using Surprise Library

In [9]:
!pip install scikit-surprise

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 26.1 MB/s eta 0:00:00


# Loading the data

In [10]:
from surprise import Dataset
from surprise import Reader
from surprise import SVD

reader = Reader(rating_scale=(0.5, 5))

data = Dataset.load_from_df(
    ratings[['userId','movieId','rating']],
    reader
)

# Train_test_split

In [11]:

from surprise.model_selection import train_test_split

trainset, testset = train_test_split(
    data,
    test_size=0.2
)

# Train SVD model

In [12]:

model = SVD()

model.fit(trainset)

# Prediction

In [13]:
predictions = model.test(testset)

predictions[:5]

[Prediction(uid=144, iid=1193, r_ui=4.0, est=np.float64(4.069051310889016), details={'was_impossible': False}),
 Prediction(uid=364, iid=733, r_ui=4.0, est=np.float64(4.155092266939775), details={'was_impossible': False}),
 Prediction(uid=452, iid=1617, r_ui=5.0, est=np.float64(4.985784277375659), details={'was_impossible': False}),
 Prediction(uid=512, iid=337, r_ui=3.0, est=np.float64(3.907999360211409), details={'was_impossible': False}),
 Prediction(uid=121, iid=337, r_ui=4.0, est=np.float64(3.4154100429906267), details={'was_impossible': False})]

#RMSC Evaluating

In [14]:
from surprise import accuracy

accuracy.rmse(predictions)

RMSE: 0.8756


np.float64(0.8756426639838656)

# Hyperparameter Tuning

In [15]:

from surprise.model_selection import GridSearchCV

param_grid = {
    'n_factors':[50,100,150],
    'n_epochs':[20,30],
    'lr_all':[0.002,0.005],
    'reg_all':[0.02,0.1]
}

gs = GridSearchCV(
    SVD,
    param_grid,
    measures=['rmse'],
    cv=3
)

gs.fit(data)

print(gs.best_score['rmse'])
print(gs.best_params['rmse'])

0.8700008943475585
{'n_factors': 100, 'n_epochs': 30, 'lr_all': 0.005, 'reg_all': 0.1}


# Save model using pickle.

In [16]:
import pickle

pickle.dump(
    model,
    open('movie_model.pkl','wb')
)